In [1]:
from stemOrchestrator.logging_config import setup_logging

data_folder = "."
out_path = data_folder
setup_logging(out_path=out_path)

In [2]:
from stemOrchestrator.acquisition import TFacquisition, DMacquisition
from stemOrchestrator.simulation import DMtwin
from autoscript_tem_microscope_client.enumerations import EdsDetectorType
from stemOrchestrator.process import HAADF_tiff_to_png, tiff_to_png
from autoscript_tem_microscope_client import TemMicroscopeClient
import matplotlib.pyplot as plt
import logging

plot = plt
from typing import Dict
import os

import numpy as np
import json
from pathlib import Path

In [3]:
ip = os.getenv("MICROSCOPE_IP")
port = os.getenv("MICROSCOPE_PORT")

if not ip or not port:
    secret_path = Path("../../config_secret.json")
    if secret_path.exists():
        with open(secret_path, "r") as f:
            secret = json.load(f)
            ip = ip or secret.get("ip_TF_sim")
            port = port or secret.get("port_TF_sim")

if not ip:
    ip = input("Enter microscope IP: ")
if not port:
    port = input("Enter microscope Port: ")
port = int(port)



In [4]:
ip

'10.46.217.242'

In [ ]:
# # from autoscript_tem_microscope_client import TemMicroscopeClient
# from autoscript_tem_microscope_client.enumerations import EdsDetectorType, ImageSize, DetectorType
# from autoscript_tem_microscope_client.structures import EdsSpectrumImageSettings

# microscope = TemMicroscopeClient()
# microscope.connect(ip, port=port)

# # 1. Retrieve the EDS detector
# eds_detector = microscope.detectors.get_eds_detector(EdsDetectorType.SUPER_X)

# # 2. Configure Spectrum Image settings (requires dwell_time, shaping_time, and dispersion)
# settings = EdsSpectrumImageSettings(
#     eds_detector=eds_detector.name,
#     dispersion=eds_detector.dispersions,
#     shaping_time=eds_detector.shaping_times,
#     dwell_time=4e-6,
#     size=ImageSize.PRESET_256, 
#     scanning_detectors=[DetectorType.HAADF] # Simultaneously grab a HAADF image
# )

# # 3. Acquire the map (This returns an AdornedSpectrumImage handle, not the raw data)
# print("Acquiring spectrum image...")
# eds_spectrum_image = microscope.analysis.eds.acquire_spectrum_image(settings)

# # 4. Save directly as an EMD file
# # This transfers the .emd file from the microscope server to your client PC
# save_path = r"C:\temp\spectrum_image.emd"
# eds_spectrum_image.save(save_path) #
# print(f"Successfully saved to {save_path}")

Client connecting to [10.46.217.242:9090]...
Client connected to [10.46.217.242:9090]


ApplicationServerException: An unexpected error occurred in the application server.
Valid Analytical license is needed for this functionality.

In [6]:
from autoscript_tem_microscope_client import TemMicroscopeClient
from autoscript_tem_microscope_client.enumerations import EdsDetectorType, ExposureTimeType
from autoscript_tem_microscope_client.structures import EdsAcquisitionSettings
import numpy as np
import hyperspy.api as hs

# # Connect to the microscope
# microscope = TemMicroscopeClient()
# microscope.connect("localhost")

# 1. Retrieve the EDS detector (e.g., Super-X)
eds_detector = microscope.detectors.get_eds_detector(EdsDetectorType.SUPER_X) [4]

# 2. Configure acquisition settings for a single spectrum
settings = EdsAcquisitionSettings()
settings.eds_detector = eds_detector.name
settings.dispersion = eds_detector.dispersions[-1]       # Set energy range per channel
settings.shaping_time = eds_detector.shaping_times[-1]   # Set pulse shaping time
settings.exposure_time = 5.0                             # 5 seconds
settings.exposure_time_type = ExposureTimeType.LIVE_TIME # Pauses timer during dead time [5]

# 3. Acquire the single spectrum
print("Acquiring point spectrum...")
spectrum = microscope.analysis.eds.acquire_spectrum(settings) [1]

# --- SAVING OPTIONS ---

# Option A: Save as a standard NumPy array (.npy)
np.save(r"C:\AutoScript_Data\point_spectrum.npy", spectrum.data) [6]
print("Saved as NumPy array.")

# Option B: Save as a simple CSV file (easily readable in Excel)
np.savetxt(r"C:\AutoScript_Data\point_spectrum.csv", spectrum.data, delimiter=",")
print("Saved as CSV.")

# Option C: Use HyperSpy to save as HDF5 or MSA format (Scientific standard)
signal = hs.signals.Signal1D(spectrum.data) [6]

# Add basic metadata so HyperSpy knows the energy scale
signal.axes_manager.name = "Energy"
signal.axes_manager.scale = spectrum.metadata.analytical_detector.dispersion [7]
signal.axes_manager.offset = spectrum.metadata.analytical_detector.offset_energy [7]
signal.axes_manager.units = "eV"

signal.save(r"C:\AutoScript_Data\point_spectrum.hdf5")
print("Saved as HDF5 via HyperSpy.")


2026-02-26 17:01:20,225 - INFO - Enabling extension exspy


ApplicationServerException: An unexpected error occurred in the application server.
Valid Analytical license is needed for this functionality.